<a href="https://colab.research.google.com/github/ShubhamP1028/Spiking-Neural-Networks/blob/main/train_deep_snn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Project A.D.I.T.I. — Deep Convolutional SNN for Plant Disease Detection
**AGRICULTURAL DISEASE INFERENCE via TEMPORAL INTELLIGENCE**

This notebook implements a deep multi-layer Spiking Neural Network (SNN) using
Leaky Integrate-and-Fire (LIF) neurons with convolutional layers for image
classification on the PlantVillage dataset (38 classes).

**Architecture**: 5 Conv blocks (32→64→128→256→512) + 2 FC layers, all with LIF neurons

**Target**: ≥90% validation accuracy with strong per-class precision

## 1. Install Dependencies

In [1]:
!pip install snntorch --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 5.4 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import json
import time
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
from torchvision import datasets, transforms

import snntorch
from snntorch import surrogate
from snntorch import functional as SF
from snntorch import spikeplot as splt

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    accuracy_score,
)

print(f"PyTorch version: {torch.__version__}")
print(f"snntorch version: {snntorch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

PyTorch version: 2.10.0+cu128
snntorch version: 0.9.4
CUDA available: True
GPU: Tesla T4
Using device: cuda


## 3. Clone Dataset

In [13]:
if not os.path.exists("PlantVillage-Dataset"):
    !git clone https://github.com/spMohanty/PlantVillage-Dataset.git
    print("✅ Dataset cloned successfully!")
else:
    print("✅ Dataset already exists.")

DATA_DIR = "PlantVillage-Dataset/raw/segmented"
print(f"Dataset path: {DATA_DIR}")
print(f"Number of classes: {len(os.listdir(DATA_DIR))}")
for i, cls in enumerate(sorted(os.listdir(DATA_DIR))):
    count = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f"  {i+1:2d}. {cls}: {count} images")

✅ Dataset already exists.
Dataset path: PlantVillage-Dataset/raw/segmented
Number of classes: 38
   1. Apple___Apple_scab: 630 images
   2. Apple___Black_rot: 621 images
   3. Apple___Cedar_apple_rust: 275 images
   4. Apple___healthy: 1645 images
   5. Blueberry___healthy: 1502 images
   6. Cherry_(including_sour)___Powdery_mildew: 1052 images
   7. Cherry_(including_sour)___healthy: 854 images
   8. Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot: 513 images
   9. Corn_(maize)___Common_rust_: 1192 images
  10. Corn_(maize)___Northern_Leaf_Blight: 985 images
  11. Corn_(maize)___healthy: 1162 images
  12. Grape___Black_rot: 1180 images
  13. Grape___Esca_(Black_Measles): 1384 images
  14. Grape___Leaf_blight_(Isariopsis_Leaf_Spot): 1076 images
  15. Grape___healthy: 423 images
  16. Orange___Haunglongbing_(Citrus_greening): 5507 images
  17. Peach___Bacterial_spot: 2297 images
  18. Peach___healthy: 360 images
  19. Pepper,_bell___Bacterial_spot: 997 images
  20. Pepper,_bell___hea

## 4. Data Pipeline with Augmentation

In [14]:
# ── Hyperparameters ──
IMG_SIZE = 128
BATCH_SIZE = 50
NUM_WORKERS = 2
SEED = 42
TRAIN_SPLIT = 0.8

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Transforms ──
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# ── Load full dataset (we apply transforms per-subset later) ──
full_dataset = datasets.ImageFolder(root=DATA_DIR)
num_classes = len(full_dataset.classes)
class_names = full_dataset.classes
print(f"\n📊 Total images: {len(full_dataset)}")
print(f"📊 Number of classes: {num_classes}")

# ── Split ──
train_size = int(TRAIN_SPLIT * len(full_dataset))
val_size = len(full_dataset) - train_size
train_subset, val_subset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# ── Apply different transforms to train/val ──
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = TransformSubset(train_subset, train_transform)
val_dataset = TransformSubset(val_subset, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f"📊 Train samples: {len(train_dataset)}")
print(f"📊 Val samples:   {len(val_dataset)}")
print(f"📊 Train batches: {len(train_loader)}")
print(f"📊 Val batches:   {len(val_loader)}")


📊 Total images: 54306
📊 Number of classes: 38
📊 Train samples: 43444
📊 Val samples:   10862
📊 Train batches: 869
📊 Val batches:   218


## 5. Deep Convolutional SNN Model

In [15]:
class DeepSNN(nn.Module):
    """
    Deep Convolutional Spiking Neural Network with LIF neurons.

    Architecture:
        5 Conv blocks (3→32→64→128→256→512) with BatchNorm + LIF + MaxPool
        2 FC layers (512*4*4 → 256 → num_classes) with LIF neurons
        Dropout for regularization
    """

    def __init__(self, num_classes=38, beta=0.95, num_steps=30):
        super(DeepSNN, self).__init__()
        self.num_steps = num_steps
        spike_grad = surrogate.fast_sigmoid(slope=25)

        # ── Block 1: 3 → 32, 128→64 ──
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.lif1 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)

        # ── Block 2: 32 → 64, 64→32 ──
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.lif2 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)

        # ── Block 3: 64 → 128, 32→16 ──
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.lif3 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)

        # ── Block 4: 128 → 256, 16→8 ──
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2)
        self.lif4 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)

        # ── Block 5: 256 → 512, 8→4 (adaptive) ──
        self.conv5 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.bn5 = nn.BatchNorm2d(512)
        self.pool5 = nn.AdaptiveAvgPool2d(4)
        self.lif5 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)

        # ── Classifier ──
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(512 * 4 * 4, 256)
        self.lif6 = snntorch.Leaky(beta=beta, spike_grad=spike_grad)
        self.fc2 = nn.Linear(256, num_classes)
        self.lif7 = snntorch.Leaky(beta=beta, spike_grad=spike_grad, output=True)

    def forward(self, x):
        # Initialize membrane potentials
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()
        mem5 = self.lif5.init_leaky()
        mem6 = self.lif6.init_leaky()
        mem7 = self.lif7.init_leaky()

        spk_rec = []
        mem_rec = []

        for step in range(self.num_steps):
            # Block 1
            cur = self.pool1(self.bn1(self.conv1(x)))
            spk1, mem1 = self.lif1(cur, mem1)

            # Block 2
            cur = self.pool2(self.bn2(self.conv2(spk1)))
            spk2, mem2 = self.lif2(cur, mem2)

            # Block 3
            cur = self.pool3(self.bn3(self.conv3(spk2)))
            spk3, mem3 = self.lif3(cur, mem3)

            # Block 4
            cur = self.pool4(self.bn4(self.conv4(spk3)))
            spk4, mem4 = self.lif4(cur, mem4)

            # Block 5
            cur = self.pool5(self.bn5(self.conv5(spk4)))
            spk5, mem5 = self.lif5(cur, mem5)

            # Classifier
            cur = self.flatten(spk5)
            cur = self.dropout(cur)
            cur = self.fc1(cur)
            spk6, mem6 = self.lif6(cur, mem6)

            cur = self.fc2(spk6)
            spk7, mem7 = self.lif7(cur, mem7)

            spk_rec.append(spk7)
            mem_rec.append(mem7)

        return torch.stack(spk_rec, dim=0), torch.stack(mem_rec, dim=0)

# ── Instantiate ──
NUM_STEPS = 30
model = DeepSNN(num_classes=num_classes, beta=0.95, num_steps=NUM_STEPS).to(device)

# ── Print Model Summary ──
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🧠 DeepSNN Model Summary")
print(f"{'='*50}")
print(f"Total parameters:     {total_params:>12,}")
print(f"Trainable parameters: {trainable_params:>12,}")
print(f"Timesteps:            {NUM_STEPS:>12}")
print(f"LIF beta:             {'0.95':>12}")
print(f"{'='*50}")
print(model)


🧠 DeepSNN Model Summary
Total parameters:        3,677,734
Trainable parameters:    3,677,734
Timesteps:                      30
LIF beta:                     0.95
DeepSNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lif1): Leaky()
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lif2): Leaky()
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (lif3): Leaky()
  (conv4): Conv

## 6. Training

In [ ]:
# ── Training Configuration ──
EPOCHS = 5
LR = 0.005
WEIGHT_DECAY = 0.0001

loss_fn = SF.ce_count_loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = GradScaler()

# ── Training History ──
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "epoch_time": [],
    "lr": [],
}

best_val_acc = 0.0
best_model_state = None

print(f"\n🚀 Starting Training")
print(f"{'='*60}")
print(f"Epochs: {EPOCHS} | LR: {LR} | Optimizer: AdamW")
print(f"Loss: CE Count Loss | Scheduler: CosineAnnealing")
print(f"Mixed Precision: Enabled")
print(f"{'='*60}\n")

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # ── Train Phase ──
    model.train()
    running_loss = 0.0
    train_batches = 0

    for batch_idx, (data, targets) in enumerate(train_loader):
        data, targets = data.to(device, non_blocking=True), targets.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            spk_rec, mem_rec = model(data)
            loss = loss_fn(spk_rec, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        train_batches += 1

        if (batch_idx + 1) % 500 == 0:
            print(f"  Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | "
                  f"Loss: {loss.item():.4f}")

    avg_train_loss = running_loss / train_batches

    # ── Validation Phase ──
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    val_batches = 0

    with torch.no_grad():
        for data, targets in val_loader:
            data, targets = data.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            spk_rec, mem_rec = model(data)
            loss = loss_fn(spk_rec, targets)
            val_loss += loss.item()
            val_batches += 1

            # Spike count accuracy
            predicted = spk_rec.sum(dim=0).argmax(dim=1)
            correct += (predicted == targets).sum().item()
            total += targets.size(0)

    avg_val_loss = val_loss / val_batches
    val_acc = 100.0 * correct / total

    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # Save history
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["val_accuracy"].append(val_acc)
    history["epoch_time"].append(epoch_time)
    history["lr"].append(current_lr)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        torch.save(best_model_state, "best_deep_snn.pth")
        marker = " ⭐ BEST"
    else:
        marker = ""

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}% | "
          f"LR: {current_lr:.6f} | "
          f"Time: {epoch_time:.1f}s{marker}")

print(f"\n{'='*60}")
print(f"🏆 Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"💾 Best model saved to: best_deep_snn.pth")

# Save training history
with open("training_history.json", "w") as f:
    json.dump(history, f, indent=2)
print(f"📊 Training history saved to: training_history.json")


🚀 Starting Training
Epochs: 5 | LR: 0.005 | Optimizer: AdamW
Loss: CE Count Loss | Scheduler: CosineAnnealing
Mixed Precision: Enabled



/tmp/ipykernel_580/284689169.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_580/284689169.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 1/5 | Batch 500/869 | Loss: 7.6965
Epoch [01/5] | Train Loss: 7.8440 | Val Loss: 4.8589 | Val Acc: 1.13% | LR: 0.005000 | Time: 934.5s ⭐ BEST
  Epoch 2/5 | Batch 500/869 | Loss: 3.5835
Epoch [02/5] | Train Loss: 4.5009 | Val Loss: 4.4285 | Val Acc: 1.13% | LR: 0.004523 | Time: 934.4s
  Epoch 3/5 | Batch 500/869 | Loss: 3.5835


## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── Loss Curves ──
axes[0].plot(range(1, EPOCHS+1), history["train_loss"], "b-o", label="Train Loss", markersize=4)
axes[0].plot(range(1, EPOCHS+1), history["val_loss"], "r-s", label="Val Loss", markersize=4)
axes[0].set_xlabel("Epoch", fontsize=12)
axes[0].set_ylabel("Loss", fontsize=12)
axes[0].set_title("Training & Validation Loss", fontsize=14, fontweight="bold")
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ── Accuracy Curve ──
axes[1].plot(range(1, EPOCHS+1), history["val_accuracy"], "g-^", label="Val Accuracy", markersize=4)
axes[1].axhline(y=90, color="red", linestyle="--", alpha=0.7, label="90% Target")
axes[1].set_xlabel("Epoch", fontsize=12)
axes[1].set_ylabel("Accuracy (%)", fontsize=12)
axes[1].set_title("Validation Accuracy", fontsize=14, fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# ── Learning Rate Schedule ──
axes[2].plot(range(1, EPOCHS+1), history["lr"], "m-d", markersize=4)
axes[2].set_xlabel("Epoch", fontsize=12)
axes[2].set_ylabel("Learning Rate", fontsize=12)
axes[2].set_title("Learning Rate Schedule", fontsize=14, fontweight="bold")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: training_curves.png")

## 8. Comprehensive Evaluation on Validation Set

In [ ]:
# Load best model
model.load_state_dict(torch.load("best_deep_snn.pth"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for data, targets in val_loader:
        data, targets = data.to(device), targets.to(device)
        spk_rec, _ = model(data)
        predicted = spk_rec.sum(dim=0).argmax(dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(targets.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# ── Overall Metrics ──
overall_acc = accuracy_score(all_labels, all_preds) * 100
precision, recall, f1, support = precision_recall_fscore_support(
    all_labels, all_preds, average="macro"
)
print(f"\n{'='*60}")
print(f"📊 EVALUATION RESULTS (Best Model)")
print(f"{'='*60}")
print(f"Overall Accuracy:   {overall_acc:.2f}%")
print(f"Macro Precision:    {precision:.4f}")
print(f"Macro Recall:       {recall:.4f}")
print(f"Macro F1-Score:     {f1:.4f}")
print(f"{'='*60}\n")

# ── Per-Class Report ──
report = classification_report(
    all_labels, all_preds,
    target_names=class_names,
    digits=4,
    output_dict=True,
)
print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

## 9. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(22, 18))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, ax=ax,
)
ax.set_xlabel("Predicted", fontsize=14)
ax.set_ylabel("True", fontsize=14)
ax.set_title("Confusion Matrix — Deep SNN (Best Model)", fontsize=16, fontweight="bold")
plt.xticks(rotation=90, ha="right", fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: confusion_matrix.png")

## 10. Per-Class Performance Charts

In [ ]:
# ── Extract per-class metrics ──
per_class_data = []
for cls_name in class_names:
    if cls_name in report:
        per_class_data.append({
            "Class": cls_name,
            "Precision": report[cls_name]["precision"],
            "Recall": report[cls_name]["recall"],
            "F1-Score": report[cls_name]["f1-score"],
            "Support": report[cls_name]["support"],
        })

df_metrics = pd.DataFrame(per_class_data)
df_metrics = df_metrics.sort_values("F1-Score", ascending=True).reset_index(drop=True)

# ── Per-Class F1 Bar Chart ──
fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.cm.RdYlGn(df_metrics["F1-Score"].values)
bars = ax.barh(df_metrics["Class"], df_metrics["F1-Score"], color=colors, edgecolor="gray", linewidth=0.5)
ax.axvline(x=0.9, color="red", linestyle="--", alpha=0.7, label="90% Target")
ax.set_xlabel("F1-Score", fontsize=13)
ax.set_title("Per-Class F1-Score — Deep SNN", fontsize=15, fontweight="bold")
ax.legend(fontsize=11)
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: per_class_f1.png")

# ── Precision vs Recall Scatter ──
fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(
    df_metrics["Precision"], df_metrics["Recall"],
    c=df_metrics["F1-Score"], cmap="RdYlGn", s=80,
    edgecolors="black", linewidth=0.5, alpha=0.8,
)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("Precision", fontsize=13)
ax.set_ylabel("Recall", fontsize=13)
ax.set_title("Precision vs Recall per Class", fontsize=15, fontweight="bold")
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("F1-Score", fontsize=12)
ax.set_xlim(0, 1.05)
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("precision_recall_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: precision_recall_scatter.png")

# ── Grouped Bar Chart: Precision, Recall, F1 ──
fig, ax = plt.subplots(figsize=(18, 8))
x = np.arange(len(df_metrics))
width = 0.25
ax.bar(x - width, df_metrics["Precision"], width, label="Precision", color="#3498db", alpha=0.85)
ax.bar(x, df_metrics["Recall"], width, label="Recall", color="#2ecc71", alpha=0.85)
ax.bar(x + width, df_metrics["F1-Score"], width, label="F1-Score", color="#e74c3c", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(df_metrics["Class"], rotation=90, ha="right", fontsize=7)
ax.set_ylabel("Score", fontsize=13)
ax.set_title("Per-Class Precision / Recall / F1-Score", fontsize=15, fontweight="bold")
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Saved: per_class_metrics.png")

## 11. Training Summary

In [11]:
print(f"\n{'='*60}")
print(f"🏆 TRAINING COMPLETE — Deep Convolutional SNN")
print(f"{'='*60}")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
print(f"Macro Precision:          {precision:.4f}")
print(f"Macro Recall:             {recall:.4f}")
print(f"Macro F1-Score:           {f1:.4f}")
print(f"Total Training Time:      {sum(history['epoch_time']):.1f}s "
      f"({sum(history['epoch_time'])/60:.1f} min)")
print(f"{'='*60}")
print(f"\n📁 Saved Files:")
print(f"  • best_deep_snn.pth          — Best model checkpoint")
print(f"  • training_history.json       — Training metrics per epoch")
print(f"  • training_curves.png         — Loss/Accuracy/LR plots")
print(f"  • confusion_matrix.png        — Confusion matrix heatmap")
print(f"  • per_class_f1.png            — Per-class F1 bar chart")
print(f"  • precision_recall_scatter.png— Precision vs Recall scatter")
print(f"  • per_class_metrics.png       — Grouped bar chart")


🏆 TRAINING COMPLETE — Deep Convolutional SNN
Best Validation Accuracy: 1.20%
Macro Precision:          0.0003
Macro Recall:             0.0263
Macro F1-Score:           0.0006
Total Training Time:      4807.2s (80.1 min)

📁 Saved Files:
  • best_deep_snn.pth          — Best model checkpoint
  • training_history.json       — Training metrics per epoch
  • training_curves.png         — Loss/Accuracy/LR plots
  • confusion_matrix.png        — Confusion matrix heatmap
  • per_class_f1.png            — Per-class F1 bar chart
  • precision_recall_scatter.png— Precision vs Recall scatter
  • per_class_metrics.png       — Grouped bar chart


## 12. Download Model Locally

In [ ]:
from google.colab import files

print("📥 Downloading best model checkpoint...")
files.download("best_deep_snn.pth")

print("📥 Downloading training history...")
files.download("training_history.json")

print("📥 Downloading plots...")
for plot_file in ["training_curves.png", "confusion_matrix.png",
                   "per_class_f1.png", "precision_recall_scatter.png",
                   "per_class_metrics.png"]:
    if os.path.exists(plot_file):
        files.download(plot_file)
        print(f"  ✅ {plot_file}")

print("\n🎉 All files downloaded!")